# 16. Other splits of the outlets (CPU)

How much the results depend on the split: the ridge models of step 7 (TF-IDF, Qwen3-Embedding-8B, Mistral-Small-24B)
again with four more splits of the outlets, made with the procedure of step 4 (folds stratified on the exact score)
and the seeds 2–5; the paper's split is seed 1. Same articles, frozen vectors of step 5 and inner folds as in step 7;
only the predictions are saved (`splits/seed<s>/<model>/`). Details in `splits.py`.

In [1]:
SEEDS = "2,3,4,5"
N_JOBS = 16

In [2]:
# Parameters
N_JOBS = 8


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO
import splits as SP

art, outlets = MO.load()
assert (MO.score_split(outlets.score, C.SPLIT_SEED) == outlets.fold).all(), "the procedure does not give the paper's split"
print("the procedure gives the paper's split with seed", C.SPLIT_SEED)

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


the procedure gives the paper's split with seed 1


In [4]:
for seed in map(int, SEEDS.split(",")):
    SP.run(seed, art, outlets, n_jobs=N_JOBS, logfile=LOG)

2026-11-17 12:09:46  split seed 2: outlets per fold [46, 45, 45, 45, 45]


2026-11-17 12:12:50  regression, TF-IDF, fold 1: penalty 0.3 (inside the grid), 185 s


2026-11-17 12:16:10  regression, TF-IDF, fold 2: penalty 0.1 (inside the grid), 200 s


2026-11-17 12:19:16  regression, TF-IDF, fold 3: penalty 0.001 (on the grid edge), 187 s


2026-11-17 12:22:28  regression, TF-IDF, fold 4: penalty 0.3 (inside the grid), 192 s


2026-11-17 12:25:41  regression, TF-IDF, fold 5: penalty 0.1 (inside the grid), 193 s


... (101 outputs left out)

2026-11-17 13:22:42  regression, Mistral-24B, fold 4: penalty 100000 (inside the grid), layer 27, 25 s


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.759455821267693e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.463497718797953e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.303605708055329e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


2026-11-17 13:23:07  regression, Mistral-24B, fold 5: penalty 100000 (inside the grid), layer 27, 25 s


In [5]:
S = SP.results(outlets)
S.to_csv(C.OUT / "paper_splits.csv", index=False)
display(S.round(3))
S.groupby("model")[["Spearman", "MAE", "Spearman - TF-IDF", "MAE - TF-IDF"]].agg(["mean", "std", "min", "max"]).round(3)

,split seed,model,Spearman,MAE,Spearman - TF-IDF,MAE - TF-IDF
0,1,TF-IDF,0.674,17.825,0.000,0.000
1,1,Qwen3-8B,0.711,16.311,0.037,-1.514
2,1,Mistral-24B,0.735,15.793,0.060,-2.031
3,2,TF-IDF,0.690,17.883,0.000,0.000
4,2,Qwen3-8B,0.696,16.272,0.007,-1.610
5,2,Mistral-24B,0.720,15.662,0.030,-2.221
6,3,TF-IDF,0.660,18.220,0.000,0.000
7,3,Qwen3-8B,0.688,16.471,0.027,-1.749
8,3,Mistral-24B,0.716,15.966,0.056,-2.254
9,4,TF-IDF,0.673,18.042,0.000,0.000


Spearman                          MAE                        Spearman - TF-IDF                      MAE - TF-IDF                     
                mean    std    min    max    mean    std     min     max              mean    std    min    max         mean    std    min    max
model                                                                                                                                            
Mistral-24B    0.727  0.011  0.716  0.742  15.850  0.166  15.662  16.071             0.053  0.015  0.030  0.069       -2.176  0.110 -2.285 -2.031
Qwen3-8B       0.700  0.011  0.688  0.713  16.343  0.151  16.146  16.517             0.026  0.013  0.007  0.040       -1.682  0.146 -1.896 -1.514
TF-IDF         0.674  0.010  0.660  0.690  18.026  0.171  17.825  18.220             0.000  0.000  0.000  0.000        0.000  0.000  0.000  0.000